In [36]:
!pip install selenium

  Using cached urllib3-2.6.2-py3-none-any.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 165.4 kB/s  0:01:06 eta 0:00:04
Using cached urllib3-2.6.2-py3-none-any.whl (131 kB)
  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.26.5
    Uninstalling urllib3-1.26.5:
      Successfully uninstalled urllib3-1.26.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [selenium]/10 [selenium]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pycaret 3.3.2 requires matplotlib<3.8.0, but you have matplotlib 3.10.8 which is incompatible.
understatapi 0.7.0 requires urllib3==1.26.5, but you have urllib3 2.6.2 which is incompatible.


In [119]:
from understatapi import UnderstatClient
import pandas as pd
import requests

from thefuzz import process, fuzz

import requests
import pandas as pd
import json
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import os
import io

gw = 21

In [66]:
players_raw = pd.read_csv('https://raw.githubusercontent.com/ilyandho/FPL-Optimal-Transfer/refs/heads/FantasyGo/FPL%20predictors/with%20new%20features/data/vaastav/data/2025-26/players_raw.csv')
players_1 = pd.read_csv('https://raw.githubusercontent.com/ilyandho/FPL-Optimal-Transfer/refs/heads/FantasyGo/FPL%20predictors/with%20new%20features/data/vaastav/data/2025-26/gws/gw1.csv')
v = pd.read_csv('https://github.com/ilyandho/FPL-Optimal-Transfer/raw/refs/heads/FantasyGo/FPL%20predictors/with%20new%20features/data/vaastav/data/2025-26/players/Aaron_Hickey_116/gw.csv')
master_fpl_understat_map = pd.read_csv('https://raw.githubusercontent.com/ChrisMusson/FPL-ID-Map/main/Master.csv')

# player_gw_stats = pd.read_csv('https://github.com/olbauday/FPL-Core-Insights/raw/refs/heads/main/data/2025-2026/By%20Tournament/Premier%20League/GW1/player_gameweek_stats.csv')
# playerstats_1 = pd.read_csv('https://github.com/olbauday/FPL-Core-Insights/raw/refs/heads/main/data/2025-2026/By%20Tournament/Premier%20League/GW1/playerstats.csv')
players = pd.read_csv('https://github.com/olbauday/FPL-Core-Insights/raw/refs/heads/main/data/2025-2026/By%20Tournament/Premier%20League/GW1/players.csv')
teams = pd.read_csv('https://github.com/olbauday/FPL-Core-Insights/raw/refs/heads/main/data/2025-2026/By%20Tournament/Premier%20League/GW1/teams.csv')

git_data_olbauday_base = "https://github.com/olbauday/FPL-Core-Insights/raw/refs/heads/main/data/2025-2026/By%20Tournament/Premier%20League/"

fpl_fixtures_url = 'https://fantasy.premierleague.com/api/fixtures/?event='

player_summary = 'https://fantasy.premierleague.com/api/element-summary/'

# bootstrap = requests.get('https://fantasy.premierleague.com/api/bootstrap-static/').json()
# bootstrap

## Understat


In [ ]:
# Initialize the client
with UnderstatClient() as understat:
    # Use .league() to specify the league, then .get_player_data() for the season
    data = understat.league(league="EPL").get_player_data(season="2025")

# This will return a list of dictionaries containing player stats
understat_data = pd.DataFrame(data)
understat_data.to_csv(f'./data/understat/understat_data_{gw}.csv')
understat_data

,id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,yellow_cards,red_cards,position,team_title,npg,npxG,xGChain,xGBuildup
0,8260,Erling Haaland,20,1756,19,18.415004886686802,4,3.0899876076728106,75,11,0,0,F,Manchester City,18,16.89266712218523,20.211640633642673,2.778261484578252
1,13222,Thiago,20,1680,14,13.802449688315392,1,1.5831863638013601,44,9,3,0,F S,Brentford,9,9.235436581075191,11.383689273148775,2.6555524803698063
2,11363,Antoine Semenyo,19,1710,9,7.810670031234622,3,2.4660277236253023,47,25,5,0,M,Bournemouth,8,6.288332333788276,10.328016273677349,2.9676714949309826
3,501,Danny Welbeck,19,1107,8,6.669347804039717,0,0.31755480915308,28,10,3,0,F S,Brighton,7,4.385841159150004,6.039988946169615,1.8120669340714812
4,5555,Dominic Calvert-Lewin,18,1224,8,7.4578270222991705,0,1.2653321214020252,36,12,0,0,F S,Leeds,7,6.696658169850707,8.989225076511502,1.3431714698672295
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480,14198,Shea Lacey,1,2,0,0.01579156517982483,0,0,1,0,0,0,S,Manchester United,0,0.01579156517982483,0.01579156517982483,0
481,14219,Mohamadou Kanté,3,8,0,0.06551869213581085,0,0.10989412665367126,1,1,0,0,S,West Ham,0,0.06551869213581085,0.17541281878948212,0
482,14263,Joél Drakes-Thomas,2,2,0,0,0,0,0,0,0,0,S,Crystal Palace,0,0,0.3753929138183594,0.3753929138183594
483,14266,Bendito Mantato,1,14,0,0,0,0.03647809848189354,0,1,0,0,S,Manchester United,0,0,0.12379638105630875,0.12379638105630875


In [ ]:

# List of IDs from your data
player_ids = understat_data['id'].values #['8260', '13222', '11363'] # Haaland, Thiago, Semenyo, etc.
player_name_id = understat_data.set_index('id')['player_name']
player_name_id
all_history = []

with UnderstatClient() as understat:
    for p_id in player_ids:
        # This gets the season-by-season history for that specific ID
        history = understat.player(player=p_id).get_match_data()

        # # Add the player name back in so you know who is who
        for gw_ in history:
            gw_['understat_id'] = p_id
            gw_['understat_name'] = player_name_id[p_id]
            all_history.append(gw_)

# # Convert to a history DataFrame
history_df = pd.DataFrame(all_history)
history_df.to_csv(f'./data/understat/understat_hist_{gw}.csv', index=False)

## FPL


### GW 1


In [ ]:
all_player_gw_stats = []
for gw in range(1,21):
    player_gw_stats = pd.read_csv(f'{git_data_olbauday_base}GW{gw}/player_gameweek_stats.csv')
    player_gw_stats['round'] = gw
    all_player_gw_stats.append(player_gw_stats)

# 2. Combine all DataFrames at once (much faster than looping concat)
all_player_gw_stats = pd.concat(all_player_gw_stats, ignore_index=True)

all_teams = []
for gw in range(1,21):
    team_stats = pd.read_csv(f'{git_data_olbauday_base}GW{gw}/teams.csv')
    team_stats['round'] = gw
    all_teams.append(team_stats)

all_team_stats = pd.concat(all_teams, ignore_index=True)

all_players = []
for gw in range(1,21):
    team_stats = pd.read_csv(f'{git_data_olbauday_base}GW{gw}/players.csv')
    team_stats['round'] = gw
    all_players.append(team_stats)

players = pd.concat(all_players, ignore_index=True)

matches = []

for gw in range(1,21):
    data = requests.get(fpl_fixtures_url+str(gw)).json()

    matches = [*matches, *[{
                    'round': event['event'], 'team_id': event['id'], 'team_a': event['team_a'], 'team_h': event['team_h'],
                    'team_h_difficulty':event['team_h_difficulty'], 'team_a_difficulty':event['team_a_difficulty'], 'team_a_score': event['team_a_score'],
                    'team_h_score': event['team_h_score'], 'kickoff_time': event['kickoff_time']
                     } for event in data]]
    #     []
    # matches.append()
match_details = pd.DataFrame(matches)
match_details.to_csv(f'./data/match_details_{gw}.csv', index=False)

In [ ]:

player_gw_stats_clean = all_player_gw_stats[all_player_gw_stats['status']!= 'u']  # Remove unavailable players
player_gw_stats_clean.columns.tolist()
gw_stats_cols =  [
                    'id', 'first_name', 'second_name', 'web_name', 'now_cost',  'selected_by_percent',  'form',  'event_points', 'transfers_in_event',
                    'transfers_out_event', 'value_form', 'ep_next', 'ep_this', 'chance_of_playing_next_round', 'chance_of_playing_this_round',  'gw',
                    'total_points', 'minutes',  'goals_scored',  'assists',  'clean_sheets',  'goals_conceded', 'yellow_cards',  'red_cards',  'saves',
                    'starts',  'bonus',  'bps',  'transfers_in',  'transfers_out', 'expected_goals',  'expected_assists',  'expected_goal_involvements',
                    'expected_goals_conceded',  'influence',  'creativity',  'threat',  'ict_index',  'tackles',  'clearances_blocks_interceptions',
                    'recoveries',  'defensive_contribution',  'round',

                    # 'expected_goals_per_90', 'expected_assists_per_90', 'expected_goal_involvements_per_90',
                    # 'expected_goals_conceded_per_90', 'saves_per_90', 'clean_sheets_per_90', 'goals_conceded_per_90', 'defensive_contribution_per_90',

                    # 'status', 'news', 'news_added', 'now_cost_rank', 'now_cost_rank_type', 'selected_rank', 'selected_rank_type', 'form_rank', 'form_rank_type',
                    # 'cost_change_event', 'cost_change_event_fall', 'cost_change_start', 'cost_change_start_fall', 'value_season', 'points_per_game',
                    # 'points_per_game_rank', 'points_per_game_rank_type', 'influence_rank', 'influence_rank_type', 'creativity_rank', 'creativity_rank_type',
                    # 'threat_rank', 'threat_rank_type', 'ict_index_rank', 'ict_index_rank_type', 'corners_and_indirect_freekicks_order', 'direct_freekicks_order',
                    # 'penalties_order', 'set_piece_threat', 'corners_and_indirect_freekicks_text', 'direct_freekicks_text', 'penalties_text',  'own_goals', '
                    # penalties_saved', 'penalties_missed', 'dreamteam_count', 'starts_per_90',
                ]

player_gw_stats_clean = player_gw_stats_clean[gw_stats_cols]
player_gw_stats_clean [['chance_of_playing_this_round', 'chance_of_playing_next_round']]= player_gw_stats_clean[['chance_of_playing_this_round', 'chance_of_playing_next_round']].fillna(100)
ids_with_details = players['player_id'].unique().tolist()
player_gw_stats_clean = player_gw_stats_clean[player_gw_stats_clean['id'].isin(ids_with_details)]
player_gw_stats_clean = player_gw_stats_clean.merge(players[['player_id','team_code', 'position', 'round']], left_on=['id', 'round'], right_on=['player_id', 'round'], how='left')
player_gw_stats_clean = player_gw_stats_clean.dropna(subset=['team_code'])
team_data = all_team_stats.rename({'code': 'team_code', 'id': 'team_id', 'name':'team', 'short_name':'team_short_name', 'ep_this': 'xP', 'ep_next':'xP_next'}, axis=1)
player_gw_stats_clean = player_gw_stats_clean.merge(team_data[[
  'team_code', 'team_id', 'team', 'round', 'team_short_name', 'elo', 'strength', 'strength_overall_home', 'strength_overall_away', 'strength_attack_home', 'strength_attack_away',
    'strength_defence_home', 'strength_defence_away']], on=['round','team_code'], how='left')

# Reshape matches so every team_id has its own row per match
match_details = match_details.rename(columns={'team_id': 'match_id'})
matches_melted = match_details.melt(
    id_vars=['match_id', 'round', 'team_h_difficulty', 'team_a_difficulty', 'team_a_score', 'team_h_score', 'kickoff_time'],
    value_vars=['team_a', 'team_h'],
    var_name='side',
    value_name='team_id'
)

player_match_df = player_gw_stats_clean.merge(matches_melted, on=['team_id', 'round'], how='left')
player_match_df

,id,first_name,second_name,web_name,now_cost,selected_by_percent,form,event_points,transfers_in_event,transfers_out_event,...,strength_attack_away,strength_defence_home,strength_defence_away,match_id,team_h_difficulty,team_a_difficulty,team_a_score,team_h_score,kickoff_time,side
0,1,David,Raya Martín,Raya,6.0,36.9,3.2,10,418466,56691,...,1350,1290,1300,9,4,3,1,0,2025-08-17T15:30:00Z,team_a
1,2,Kepa,Arrizabalaga Revuelta,Arrizabalaga,4.1,0.4,0.0,0,793,2256,...,1350,1290,1300,9,4,3,1,0,2025-08-17T15:30:00Z,team_a
2,4,Tommy,Setford,Setford,3.9,0.2,0.0,0,2397,1906,...,1350,1290,1300,9,4,3,1,0,2025-08-17T15:30:00Z,team_a
3,5,Gabriel,dos Santos Magalhães,Gabriel,6.2,14.0,0.0,6,5927,342245,...,1350,1290,1300,9,4,3,1,0,2025-08-17T15:30:00Z,team_a
4,6,William,Saliba,Saliba,6.0,10.3,0.5,9,25445,160545,...,1350,1290,1300,9,4,3,1,0,2025-08-17T15:30:00Z,team_a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12241,646,João Victor,Gomes da Silva,Gomes,5.3,0.1,2.5,1,175,239,...,1050,1090,1120,200,2,2,0,3,2026-01-03T15:00:00Z,team_h
12242,695,Jackson,Tchatchoua,Tchatchoua,4.4,0.0,1.5,5,89,43,...,1050,1090,1120,200,2,2,0,3,2026-01-03T15:00:00Z,team_h
12243,709,Ladislav,Krejcí,Krejčí,4.5,0.1,3.2,6,818,266,...,1050,1090,1120,200,2,2,0,3,2026-01-03T15:00:00Z,team_h
12244,777,Temple,Ojinnaka,Ojinnaka,4.5,0.0,0.0,0,40,10,...,1050,1090,1120,200,2,2,0,3,2026-01-03T15:00:00Z,team_h


In [ ]:
unique_players = player_gw_stats_clean['id'].unique()

player_api_data = []

for pid in unique_players:
    # Fetch from API
    response = requests.get(f'{player_summary}{pid}/')
    data = response.json()  # assuming JSON response
    history_data = data['history']

    player_api_data = [*player_api_data, *history_data]

pd.DataFrame(player_api_data).rename({'element': 'id'}, axis=1).to_csv(f'./data/player_summary_{gw}.csv', index=False)

In [ ]:
player_api_details = pd.read_csv(f'./data/player_summary_{gw}.csv')
understat_data = pd.read_csv(f'./data/understat/understat_hist_{gw}.csv')
understat_data = understat_data[understat_data['season'] == 2025]

# ids availabe from the api
live_ids = player_api_details['id'].unique().tolist()
player_match_df_ = player_match_df[player_match_df['id'].isin(live_ids)].copy()

fpl_understat_map = master_fpl_understat_map[(
                                                ~master_fpl_understat_map['25-26'].isna())
                                                & ~(master_fpl_understat_map['understat'].isna())].rename({'25-26': 'fpl_id', 'understat':'understat_id'}, axis=1)

player_match_df_['date'] = pd.to_datetime(player_match_df_['kickoff_time']).dt.date
understat_data['date'] = pd.to_datetime(understat_data['date']).dt.date
player_gw_stats_full = player_match_df_.merge(player_api_details[['id', 'round', 'selected', 'transfers_balance', 'was_home', 'value', 'opponent_team']],
                                            on=['round', 'id'],
                                            how='left')

player_gw_stats_full = player_gw_stats_full.rename(columns={'player_id':'fpl_id'}).drop('id', axis=1)

player_gw_stats_full = player_gw_stats_full[player_gw_stats_full['fpl_id'].isin(fpl_understat_map['fpl_id'].unique().tolist())]

understat_data = understat_data.merge(fpl_understat_map[['fpl_id', 'understat_id']], on='understat_id', how='left').dropna()
player_gw_stats_full = player_gw_stats_full.merge(fpl_understat_map[['fpl_id', 'understat_id']],
                                                  on='fpl_id',
                                                  how='left')

player_gw_stats_full['full_name'] = (player_gw_stats_full['first_name'] + ' ' + player_gw_stats_full['second_name']).str.strip()

player_data = player_gw_stats_full.merge(understat_data[[
                                'goals', 'shots', 'xG','h_team', 'a_team',
                                'h_goals', 'a_goals', 'date', 'season', 'roster_id', 'xA',
                                'key_passes', 'npg', 'npxG', 'xGChain', 'xGBuildup',
                                'understat_id', 'understat_name', 'fpl_id']],
                                on=['date', 'fpl_id', 'understat_id'],
                                how='left'
                                )

player_data = player_data.rename(columns={'ep_this': 'xP', 'ep_next': 'xP_next'})

player_data.to_csv('./data/player_data.csv', index=False)

### Add Odds


In [ ]:
player_data = pd.read_csv('./data/player_data.csv', low_memory=False)
player_data_clean = player_data.dropna()
# Load the current season's data directly from the source
season = "2526" # Change this for historical seasons
url = f"https://www.football-data.co.uk/mmz4281/{season}/E0.csv"

# It's good practice to use a custom User-Agent to avoid blocks
odds_data = pd.read_csv(url, low_memory=False)

# 1. Clean up dates in betting data
odds_data['Date'] = pd.to_datetime(odds_data['Date'], dayfirst=True).dt.date

# 2. Extract key columns (B365H = Bet365 Home Odds, B365D = Draw, B365A = Away)
odds_subset = odds_data[['Date', 'HomeTeam', 'AwayTeam', 'B365H', 'B365D', 'B365A']].copy()
odds_subset = odds_subset.rename(columns={'Date': 'date'})

def add_odds(row):
    win = round(1/row['B365H'], 5)
    draw = round(1/row['B365D'], 5)
    lose = round(1/row['B365A'], 5)

    # Normalize the probabilities (to make the probabilities sum to 100%)
    sum_percent = win + draw + lose
    win_prob = round(win/sum_percent, 3)
    draw_prob = round(draw/sum_percent, 3)
    lose_prob = round(lose/sum_percent, 3)

    return pd.Series([win_prob, draw_prob, lose_prob])

odds_subset[['win_prob', 'draw_prob', 'lose_prob']] = odds_subset.apply(add_odds, axis=1)

# Make sure the names match in the player_data df and odds_subset df
team_map = {
    'Bournemouth': 'Bournemouth',
    'Newcastle': 'Newcastle',
    'Fulham': 'Fulham',
    'West Ham': 'West Ham',
    'Burnley': 'Burnley',
    'Man City': 'Man City',
    'Crystal Palace': 'Crystal Palace',
    'Brentford': 'Brentford',
    'Arsenal': 'Arsenal',
    'Everton': 'Everton',
    'Chelsea': 'Chelsea',
    'Tottenham': 'Spurs',
    'Wolves': 'Wolves',
    'Aston Villa': 'Aston Villa',
    'Sunderland': 'Sunderland',
    'Leeds': 'Leeds',
    "Nott'm Forest": "Nott'm Forest",
    'Brighton': 'Brighton',
    'Man United': 'Man Utd',
    'Liverpool': 'Liverpool'
}

odds_subset['HomeTeam'] = odds_subset['HomeTeam'].map(team_map)
odds_subset['AwayTeam'] = odds_subset['AwayTeam'].map(team_map)

odds_melted = odds_subset.melt(
    id_vars=['date','B365H', 'B365D', 'B365A', 'win_prob', 'draw_prob', 'lose_prob'],
    value_vars=['HomeTeam', 'AwayTeam'],
    var_name='side',
    value_name='team'
)

odds_melted['date'] = odds_melted['date'].astype(str)

player_data_odds = player_data_clean.merge(odds_melted[['date', 'win_prob', 'draw_prob', 'lose_prob','team']],
                  on=['date', 'team'],
                  how='left'
                  )

# 1. Sort by player and round to ensure the sequence is correct
player_data_odds = player_data_odds.sort_values(['fpl_id', 'round'])

# 2. Group by player and apply the difference
player_data_odds['ownership_change'] = player_data_odds.groupby('fpl_id')['selected'].diff().fillna(0)
player_data_odds['pts_bonus'] = player_data_odds['total_points'] - player_data_odds['bonus']

def ownership_change(row):
    net_transfers = row['transfers_in'] - row['transfers_out']
    total_transfers = row['transfers_in'] + row['transfers_out']
    net_transfers_pct = net_transfers / total_transfers if total_transfers != 0 else 0

    return net_transfers_pct

player_data_odds['percenatge_net_transfers'] = player_data_odds.apply(ownership_change, axis=1)


player_data_odds.to_csv('./data/final_player_data.csv')

### Roll values


In [18]:
player_data_odds = pd.read_csv('./data/final_player_data.csv')

rolling_features = [
    'creativity', 'influence', 'threat', 'minutes',  'pts_bonus', 'total_points',
    'expected_goals', 'expected_assists', 'xP',
    'expected_goals_conceded', 'goals_conceded', 'goals_scored',
    'shots', 'key_passes', 'npg', 'npxG','goals', 'shots', 'xG','xA',
    'saves', 'starts', 'yellow_cards', 'red_cards','assists', 'clean_sheets',

    'value', 'ict_index', 'selected', 'transfers_in', 'transfers_out',
    'ownership_change', 'percenatge_net_transfers',
    'xGChain', 'xGBuildup', 'expected_goal_involvements', 'form',
    'clearances_blocks_interceptions', 'tackles', 'recoveries', 'defensive_contribution', 'selected_by_percent',
    'goals_conceded','goals_scored',

    'strength', 'strength_overall_home', 'strength_overall_away',
    'strength_attack_home', 'strength_attack_away', 'strength_defence_home', 'strength_defence_away',
]

player_df = player_data_odds.copy()


## Cater for the mssing rounds before rolling
# 1. Get all unique players and all unique rounds
all_players = player_df['fpl_id'].unique()
all_rounds = range(player_df['round'].min(), player_df['round'].max() + 1)

# 2. Create a MultiIndex of every player x every round
multi_idx = pd.MultiIndex.from_product([all_players, all_rounds], names=['fpl_id', 'round'])

# 3. Reindex the dataframe
# This inserts "empty" rows for missing player/round combinations
player_df_complete = player_df.set_index(['fpl_id', 'round']).reindex(multi_idx).reset_index()

# Fill statistical columns with 0
# stat_columns = ['goals_scored', 'goals_conceded', 'expected_goals_conceded']
player_df_complete = player_df_complete.fillna(0)

# Sort to ensure chronological order for the rolling window
player_df_complete = player_df_complete.sort_values(['fpl_id', 'round'])

# Averagae values
for col in rolling_features:
    # The rolling window now spans actual calendar rounds
    player_df_complete[f'{col}_1'] = player_df_complete.groupby(['fpl_id'])[col].shift(1).rolling(1, min_periods=1).mean().reset_index(0, drop=True)
    player_df_complete[f'{col}_3'] = player_df_complete.groupby(['fpl_id'])[col].shift(1).rolling(3, min_periods=3).mean().reset_index(0, drop=True)
    player_df_complete[f'{col}_5'] = player_df_complete.groupby(['fpl_id'])[col].shift(1).rolling(5, min_periods=5).mean().reset_index(0, drop=True)

    player_df_complete = player_df_complete.copy()

player_df_complete = player_df_complete[player_df_complete['team_id'] != 0]


# Add opponent details
# Aggregate player data so there is exactly ONE row per team per round
team_lookup = player_df_complete.groupby(['team', 'round']).agg({
    'clean_sheets': 'max',
    'elo': 'first', # Elo is usually the same for all players on the team
    'strength': 'first'
}).reset_index()

# Rename columns to 'opponent_...' so they don't clash with the player's own stats
team_lookup.columns = ['opponent_team_name', 'round'] + [f'opp_{col}' for col in team_lookup.columns if col not in ['team', 'round']]

# check your column name for who the player is playing AGAINST (e.g., 'opponent')
player_df_final = player_df_complete.merge(
    team_lookup,
    left_on=['team', 'round'],  # 'opponent' is the ID of the team they face
    right_on=['opponent_team_name', 'round'],
    how='left'
)

# numeric_cols = player_df_final.select_dtypes(include=['number']).columns

# # Tier 1: Player-level mean
# player_df_final[numeric_cols] = player_df_final.groupby('fpl_id')[numeric_cols].transform(lambda x: x.fillna(x.mean()))

# # Tier 2: Team-level mean (for players with NO personal stats yet)
# player_df_final[numeric_cols] = player_df_final.groupby(['team', 'round'])[numeric_cols].transform(lambda x: x.fillna(x.mean()))

# # Tier 3: Global constant (The safety net)
# player_df_final[numeric_cols] = player_df_final[numeric_cols].fillna(0)


player_df_final.to_csv('./data/rolled_player_data.csv', index=False)
player_df_final

,fpl_id,round,Unnamed: 0,first_name,second_name,web_name,now_cost,selected_by_percent,form,event_points,...,strength_defence_home_1,strength_defence_home_3,strength_defence_home_5,strength_defence_away_1,strength_defence_away_3,strength_defence_away_5,opponent_team_name,opp_clean_sheets,opp_elo,opp_strength
0,1.0,1,0.0,David,Raya Martín,Raya,6.0,36.9,3.2,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,Arsenal,1.0,2037.0,4.0
1,1.0,2,293.0,David,Raya Martín,Raya,5.5,21.5,8.0,6.0,...,1290.0,NaN,NaN,1300.0,NaN,NaN,Arsenal,1.0,2037.0,4.0
2,1.0,3,603.0,David,Raya Martín,Raya,5.5,21.8,6.0,2.0,...,1290.0,NaN,NaN,1300.0,NaN,NaN,Arsenal,1.0,2037.0,4.0
3,1.0,4,905.0,David,Raya Martín,Raya,5.5,23.5,6.0,6.0,...,1290.0,1290.000000,NaN,1300.0,1300.000000,NaN,Arsenal,1.0,2037.0,4.0
4,1.0,5,1209.0,David,Raya Martín,Raya,5.5,24.0,4.0,2.0,...,1290.0,1290.000000,NaN,1300.0,1300.000000,NaN,Arsenal,0.0,2037.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6022,737.0,14,4112.0,Jonah,Kusi-Asare,Kusi-Asare,4.5,0.1,0.2,1.0,...,0.0,0.000000,224.0,0.0,0.000000,224.0,Fulham,0.0,1776.0,3.0
6023,737.0,17,4980.0,Jonah,Kusi-Asare,Kusi-Asare,4.5,0.1,0.4,1.0,...,0.0,366.666667,220.0,0.0,373.333333,224.0,Fulham,1.0,1785.0,3.0
6024,737.0,20,5879.0,Jonah,Kusi-Asare,Kusi-Asare,4.5,0.1,0.3,1.0,...,0.0,366.666667,220.0,0.0,373.333333,224.0,Fulham,0.0,1798.0,3.0
6025,739.0,17,5020.0,Divine,Mukasa,Mukasa,4.4,0.0,0.2,1.0,...,0.0,0.000000,0.0,0.0,0.000000,0.0,Man City,1.0,1998.0,4.0


# Current game week


## Get Odds from WilliamHill


In [207]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

# Setup Driver (Adding options for stability)
options = webdriver.EdgeOptions()
# options.add_argument("--headless") # Uncomment to run without a window
driver = webdriver.Edge(options=options)

premLeague = "https://sports.williamhill.com/betting/en-gb/football/competitions/OB_TY295/English-Premier-League/matches/OB_MGMB/Match-Betting"
driver.get(premLeague)

# Explicit Wait: Wait up to 10 seconds for the match rows to appear
wait = WebDriverWait(driver, 10)
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "article.sp-o-market--default")))

matches = driver.find_elements(By.CSS_SELECTOR, "article.sp-o-market--default")
# details = pd.DataFrame(columns=['team_h', 'team_h', 'win_prob', 'draw_prob', 'lose_prob'])
odds_list = []
for match in matches:
    try:
        # 1. Extract teams
        teams_text = match.find_element(By.CSS_SELECTOR, 'main.sp-o-market__title span').text
        if ' v ' not in teams_text: continue

        h_team_name = teams_text.split(' v ')[0]
        a_team_name = teams_text.split(' v ')[1]

        # 2. Extract odds
        odds_els = match.find_elements(By.CSS_SELECTOR, 'section.sp-o-market__buttons .sp-betbutton > span')

        if len(odds_els) < 3: continue

        decimal_odds = []
        for btn in odds_els:
            text = btn.text.strip()
            if text == 'EVS' or text == '1/1':
                decimal_odds.append(2.0)
            elif '/' in text:
                n, d = map(int, text.split('/'))
                decimal_odds.append((n / d) + 1)
            else:
                decimal_odds.append(float(text))

        # 3. Probability Calculation
        h_raw = 1 / decimal_odds[0]
        d_raw = 1 / decimal_odds[1]
        a_raw = 1 / decimal_odds[2]

        margin_total = h_raw + d_raw + a_raw

        # print('-----------------------------------------------------')
        # 4. Append to DataFrame
        match_row = {
            'team_h_name': h_team_name,
            'team_a_name': a_team_name,
            'win_prob': round(h_raw / margin_total, 3),
            'draw_prob':round(d_raw / margin_total, 3),
            'lose_prob': round(a_raw / margin_total, 3)
        }
        odds_list.append(match_row)
        # print(match_row)

    except Exception as e:
        print(f"Skipping a match due to error: {e}")


odds_details = pd.DataFrame(odds_list)

driver.quit()

odds_details.to_csv(f'./data/odds_{gw}.csv', index=False)

odds_details

,team_h_name,team_a_name,win_prob,draw_prob,lose_prob
0,Bournemouth,Tottenham,0.436,0.269,0.295
1,Brentford,Sunderland,0.515,0.265,0.221
2,Crystal Palace,Aston Villa,0.307,0.283,0.409
3,Everton,Wolves,0.531,0.273,0.196
4,Fulham,Chelsea,0.282,0.274,0.444
5,Man City,Brighton,0.664,0.194,0.143
6,Burnley,Man Utd,0.194,0.246,0.560
7,Newcastle,Leeds,0.556,0.251,0.193
8,Arsenal,Liverpool,0.586,0.230,0.184
9,Man Utd,Man City,0.236,0.236,0.527


## FPL Live Data


In [ ]:

# 'https://fantasy.premierleague.com/api/element-summary/21'
    # 'round', 'web_name', 'position',


# 'https://fantasy.premierleague.com/api/fixtures/?event=21'
    # 'team_h_difficulty', 'team_a_difficulty', 'opponent_team', 'was_home',

# https://fantasy.premierleague.com/api/bootstrap-static/
    # elements
        # 'fpl_id',
        # chance_of_playing_next_round', 'chance_of_playing_this_round'
        # 'xP_next', 'xP',
        # 'form', 'value_form', 'value',  ',
        # 'selected_by_percent', 'transfers_in', 'transfers_out',  'selected', 'transfers_balance', 'ownership_change', 'percenatge_net_transfers',
        # 'influence', 'creativity', 'threat', 'ict_index',
    # teams
        # 'strength','strength_overall_home', 'strength_overall_away', 'strength_attack_home', 'strength_attack_away',
        # 'strength_defence_home', 'strength_defence_away',  'team_id', 'team',

# http://api.clubelo.com/2026-01-06
    # 'elo'

# WilliamHill
    # 'win_prob', 'draw_prob', 'lose_prob',

In [ ]:
current_player_summary = requests.get(f'https://fantasy.premierleague.com/api/bootstrap-static/').json()
events = pd.DataFrame(requests.get('https://fantasy.premierleague.com/api/fixtures/?event=21').json())
elo_df = pd.read_csv('http://api.clubelo.com/2026-01-06')

In [240]:
current_player_summary['elements']

[{'can_transact': True,
  'can_select': True,
  'chance_of_playing_next_round': None,
  'chance_of_playing_this_round': None,
  'code': 154561,
  'cost_change_event': 0,
  'cost_change_event_fall': 0,
  'cost_change_start': 5,
  'cost_change_start_fall': -5,
  'dreamteam_count': 1,
  'element_type': 1,
  'ep_next': '3.3',
  'ep_this': '3.8',
  'event_points': 1,
  'first_name': 'David',
  'form': '2.8',
  'id': 1,
  'in_dreamteam': False,
  'news': '',
  'news_added': None,
  'now_cost': 60,
  'photo': '154561.jpg',
  'points_per_game': '4.0',
  'removed': False,
  'second_name': 'Raya Martín',
  'selected_by_percent': '33.6',
  'special': False,
  'squad_number': None,
  'status': 'a',
  'team': 1,
  'team_code': 3,
  'total_points': 80,
  'transfers_in': 3299405,
  'transfers_in_event': 26366,
  'transfers_out': 1713405,
  'transfers_out_event': 198981,
  'value_form': '0.5',
  'value_season': '13.3',
  'web_name': 'Raya',
  'region': 200,
  'team_join_date': '2024-07-04',
  'birth_d

In [257]:
teams_data = pd.DataFrame(current_player_summary['teams'])
players_data = pd.DataFrame(current_player_summary['elements'])

teams_ids = teams[['id', 'name']].rename({'id': 'team', 'name': 'team_name'}, axis=1)
teams_data = teams_data.rename({'id': 'team', 'name': 'team_name'}, axis=1)
players_data_ = players_data.merge(teams_data[[
                       'team', 'team_name', 'strength','strength_overall_home', 'strength_overall_away', 'strength_attack_home', 'strength_attack_away',
                        'strength_defence_home', 'strength_defence_away']],
                        on='team',
                        how='left'
                        )
players_data_ = players_data_.rename({'element_type': 'position', 'ep_this': 'xP', 'ep_next': 'xP_next', 'now_cost': 'value',}, axis=1)
# Get event details
events_ = events[['event', 'team_a', 'team_h', 'team_h_difficulty', 'team_a_difficulty']]

def add_team_details(row):
    team = row['team']
    event_ = events_[(events_['team_a'] == team) | (events_['team_h'] == team)]

    was_home = team == event_['team_h'].values[0]
    opponent_team = event_['team_a'].values[0] if was_home else event_['team_h'].values[0]
    event = event_['event'].values[0]

    return pd.Series([
                        event_['team_h'].values[0],
                        event_['team_a'].values[0],
                        event_['team_h_difficulty'].values[0],
                        event_['team_a_difficulty'].values[0],
                        was_home,
                        opponent_team,
                        event])

players_data_[['team_h', 'team_a', 'team_h_difficulty', 'team_a_difficulty', 'was_home', 'opponent_team', 'round']] = players_data_.apply(add_team_details, axis=1)

# Add Elo
pl_elo = elo_df[(elo_df['Country'] == 'ENG') & (elo_df['Level'] == 1)].sort_values('Elo', ascending=False)

elo_fpl_teams = {
    'Arsenal' : 'Arsenal',
    'Man City' : 'Man City',
    'Liverpool' :'Liverpool',
    'Aston Villa' : 'Aston Villa' ,
    'Chelsea' : 'Chelsea',
    'Newcastle' : 'Newcastle',
    'Brighton' : 'Brighton',
    'Man United' : 'Man Utd',
    'Brentford' : 'Brentford',
    'Tottenham' : 'Spurs',
    'Everton' : 'Everton',
    'Crystal Palace' : 'Crystal Palace',
    'Fulham' : 'Fulham',
    'Bournemouth' : 'Bournemouth',
    'Forest' : "Nott'm Forest",
    'Leeds' : 'Leeds',
    'West Ham' : 'West Ham',
    'Burnley' : 'Burnley',
    'Sunderland' : 'Sunderland',
    'Wolves' : 'Wolves'
}

pl_elo['team_name'] = pl_elo['Club'].map(elo_fpl_teams)
pl_elo = pl_elo.rename({'Elo': 'elo'}, axis=1)

players_data_ = players_data_.merge(pl_elo[['team_name', 'elo']],
                    on='team_name',
                    how='left')

# Adds odds details
odds_details = pd.read_csv('./data/odds_21.csv')
odds_details = odds_details.rename({'a_team': 'team_a_name', 'h_team': 'team_h_name'}, axis=1)

id_team_map = pd.Series(teams['name'].values, index=teams['id']).to_dict()

players_data_['team_h_name'] =  players_data_['team_h'].map(id_team_map)
players_data_['team_a_name'] =  players_data_['team_a'].map(id_team_map)

players_data_ = players_data_.merge(odds_details,
                    on=['team_h_name', 'team_a_name'],
                    how='left')
players_data_.to_csv(f'./data/player_data{gw}.csv')
players_data_ = players_data_.rename({'id': 'fpl_id', 'team': 'team_id', 'team_name': 'team'}, axis=1)

In [ ]:
players_data_[['xP', 'creativity', 'influence', 'threat', 'minutes', 'total_points',  'expected_goals', 'expected_assists', 'xP',
    'expected_goals_conceded', 'goals_conceded', 'goals_scored',
    'shots', 'key_passes', 'npg', 'npxG','goals', 'shots', 'xG','xA',
    'saves', 'starts', 'yellow_cards', 'red_cards','assists', 'clean_sheets',]]

KeyError: "['shots', 'key_passes', 'npg', 'npxG', 'goals', 'xG', 'xA'] not in index"

In [ ]:
players_data_[


    'value', 'ict_index', 'selected', 'transfers_in', 'transfers_out',
    'ownership_change', 'percenatge_net_transfers',
    'xGChain', 'xGBuildup', 'expected_goal_involvements', 'form',
    'clearances_blocks_interceptions', 'tackles', 'recoveries', 'defensive_contribution', 'selected_by_percent',
    'goals_conceded','goals_scored',

    'strength', 'strength_overall_home', 'strength_overall_away',
    'strength_attack_home', 'strength_attack_away', 'strength_defence_home', 'strength_defence_away',
]

KeyError: ('creativity', 'influence', 'threat', 'minutes', 'total_points', 'expected_goals', 'expected_assists', 'xP', 'expected_goals_conceded', 'goals_conceded', 'goals_scored', 'shots', 'key_passes', 'npg', 'npxG', 'goals', 'shots', 'xG', 'xA', 'saves', 'starts', 'yellow_cards', 'red_cards', 'assists', 'clean_sheets', 'value', 'ict_index', 'selected', 'transfers_in', 'transfers_out', 'ownership_change', 'percenatge_net_transfers', 'xGChain', 'xGBuildup', 'expected_goal_involvements', 'form', 'clearances_blocks_interceptions', 'tackles', 'recoveries', 'defensive_contribution', 'selected_by_percent', 'goals_conceded', 'goals_scored', 'strength', 'strength_overall_home', 'strength_overall_away', 'strength_attack_home', 'strength_attack_away', 'strength_defence_home', 'strength_defence_away')